# 01 · Daily quality-issue distribution

Runs every morning. Takes **yesterday's** quality issues from the operations workbook and writes, for each area manager, a folder with one formatted Excel file per branch. The field photo of each issue is embedded next to its row.

```
operations workbook ──► this notebook ──► output/managers/<Manager>/<Branch>.xlsx
```

The sample data in `sample_data/` is fake but has the same structure as the production workbook.

In [ ]:
import os
import re
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
from datetime import datetime, timedelta

import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment, Side, Border
from openpyxl.drawing.image import Image as XLImage
from openpyxl.drawing.spreadsheet_drawing import AnchorMarker, TwoCellAnchor
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings("ignore")

## Configuration

`REPORT_DATE` is the day being reported. In production it is `None`, which means yesterday. The sample data ends on 2026-10-26, so that's the default here.

In [ ]:
WORKBOOK_PATH = os.getenv("WORKBOOK_PATH", "../sample_data/sample_workbook.xlsx")
MANAGERS_PATH = os.getenv("MANAGERS_PATH", "../sample_data/managers.xlsx")
SHEET_NAME    = "Reviewing Reports"
OUTPUT_DIR    = os.getenv("OUTPUT_DIR", "../output/managers")
REPORT_DATE   = os.getenv("REPORT_DATE", "2026-10-26")   # None -> yesterday

if REPORT_DATE is None:
    REPORT_DATE = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

## Step 1 · Find each row's photo

Photos are inserted with Excel's *Place in Cell*, which stores them as **rich values** rather than drawings. pandas and openpyxl can't see them, so this function reads the `.xlsx` package directly:

`cell vm="n"` → rich value *n* → `richValueRel.xml` → `xl/media/imageN.png`

It returns `{excel_row: image_bytes}`.

In [ ]:
def get_image_order_from_rich_data(xlsx_path, sheet_name):
    """
    Correctly maps each image to its actual Excel row using the vm attribute
    from the sheet XML. Handles 1-based vm indexing and duplicate vm values.
    Returns a dict: {excel_row_num: image_bytes}
    """
    NS_PKG = 'http://schemas.openxmlformats.org/package/2006/relationships'
    NS_DOC = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'
    NS_WB  = 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'
    NS_RVR = 'http://schemas.microsoft.com/office/spreadsheetml/2022/richvaluerel'

    with zipfile.ZipFile(xlsx_path, 'r') as z:

        # ── Step 1: Locate the correct sheet XML file ──────────────────────
        wb_root = ET.parse(z.open('xl/workbook.xml')).getroot()
        sheet_rel_id = None
        for sh in wb_root.findall(f'.//{{{NS_WB}}}sheet'):
            if sh.get('name') == sheet_name:
                sheet_rel_id = sh.get(f'{{{NS_DOC}}}id')
                break
        if not sheet_rel_id:
            raise ValueError(f"Sheet '{sheet_name}' not found in workbook.")

        rels_root = ET.parse(z.open('xl/_rels/workbook.xml.rels')).getroot()
        sheet_path = None
        for rel in rels_root.findall(f'{{{NS_PKG}}}Relationship'):
            if rel.get('Id') == sheet_rel_id:
                sheet_path = 'xl/' + rel.get('Target').lstrip('/').removeprefix('xl/')
                break
        if not sheet_path:
            raise ValueError("Could not resolve sheet file path.")

        print(f"📄 Sheet file: {sheet_path}")

        # ── Step 2: Build vm_val -> [list of row numbers] ──────────────────
        # vm values are 1-based and can be duplicated (same image in multiple rows)
        sheet_root = ET.parse(z.open(sheet_path)).getroot()
        vm_to_rows = {}
        for cell in sheet_root.findall(f'.//{{{NS_WB}}}c'):
            vm = cell.get('vm')
            if vm is not None:
                row_num = int(re.search(r'\d+', cell.get('r')).group())
                vm_val  = int(vm)
                vm_to_rows.setdefault(vm_val, []).append(row_num)

        print(f"📋 Found {len(vm_to_rows)} unique vm values across "
              f"{sum(len(v) for v in vm_to_rows.values())} cells.")

        # ── Step 3: Load relationship IDs ──────────────────────────────────
        rel_elements = ET.parse(z.open('xl/richData/richValueRel.xml')) \
                         .getroot().findall(f'.//{{{NS_RVR}}}rel')
        rel_ids = [r.get(f'{{{NS_DOC}}}id') for r in rel_elements]
        print(f"📄 Found {len(rel_ids)} relationship IDs.")

        # ── Step 4: Map relationship ID -> image file path ─────────────────
        id_to_target = {
            r.get('Id'): r.get('Target')
            for r in ET.parse(z.open('xl/richData/_rels/richValueRel.xml.rels'))
                        .getroot()
                        .findall(f'{{{NS_PKG}}}Relationship')
        }
        print(f"🔗 Relationship mapping loaded.")

        # ── Step 5: Map every row to its image bytes ───────────────────────
        # vm values are 1-based → subtract 1 to get the 0-based rel_ids index
        # Multiple rows can share the same vm value → assign image to all of them
        row_image_bytes = {}
        for vm_val, rows in vm_to_rows.items():
            rel_idx = vm_val - 1  # fix: vm is 1-based
            if rel_idx < 0 or rel_idx >= len(rel_ids):
                print(f"⚠️  vm={vm_val} → index {rel_idx} out of range (0–{len(rel_ids)-1})")
                continue
            target = id_to_target.get(rel_ids[rel_idx])
            if not target:
                print(f"⚠️  No target for vm={vm_val}")
                continue
            try:
                img_bytes = z.read(target.replace('..', 'xl'))
                for row_num in rows:  # assign to ALL rows sharing this vm
                    row_image_bytes[row_num] = img_bytes
            except KeyError:
                print(f"❌ Could not read {target}")

        print(f"📊 Mapped images to {len(row_image_bytes)} rows.")
        return row_image_bytes

## Step 2 · Write one formatted branch file

Header styling, borders, wrapped and centred text, dates as `yyyy-mm-dd`, fit to one page width. Each photo is sized to its cell and centred in it.

In [ ]:
def create_formatted_excel(file_path, data_df, row_image_mapping=None, image_col_name='Images'):
    workbook = openpyxl.Workbook()
    sheet = workbook.active

    headers = list(data_df.columns)
    if image_col_name not in headers:
        headers.append(image_col_name)

    for col_num, header in enumerate(headers, start=1):
        header_cell = sheet.cell(row=1, column=col_num)
        header_cell.value = header
        header_cell.font = Font(size=10, name='Calibri', bold=True)
        header_cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    for row_num, row_data in enumerate(data_df.values, start=2):
        for col_num, cell_data in enumerate(row_data, start=1):
            if pd.isna(cell_data):
                cell_data = None
            cell = sheet.cell(row=row_num, column=col_num, value=cell_data)
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    thin_border = Border(
        left=Side(style='thin'), right=Side(style='thin'),
        top=Side(style='thin'), bottom=Side(style='thin')
    )
    for row in sheet.iter_rows(min_row=1, max_row=sheet.max_row,
                               min_col=1, max_col=sheet.max_column):
        for cell in row:
            cell.border = thin_border

    for col_num, header in enumerate(headers, start=1):
        col_letter = get_column_letter(col_num)
        # ✅ 200px ÷ 7px-per-unit ≈ 28.57 Excel column width units
        sheet.column_dimensions[col_letter].width = 28.57 if header == image_col_name else 12

    sheet.page_setup.fitToWidth = 1

    if 'Date' in headers:
        date_col_letter = get_column_letter(headers.index('Date') + 1)
        for cell in sheet[date_col_letter][1:]:
            cell.number_format = 'yyyy-mm-dd'

    if row_image_mapping:
        image_col_idx    = headers.index(image_col_name) + 1
        image_col_letter = get_column_letter(image_col_idx)

        EMU_PER_PIXEL  = 9525
        IMG_W,  IMG_H  = 206, 199   # image size — 20px padding on each side
        CELL_W, CELL_H = 204, 201   # ✅ cell is 200×200px

        # Perfect centering: equal padding on all sides
        offset_x = int((CELL_W - IMG_W) / 2 * EMU_PER_PIXEL)  # = 20px
        offset_y = int((CELL_H - IMG_H) / 2 * EMU_PER_PIXEL)  # = 20px

        for position, img in row_image_mapping.items():
            excel_row = position + 2
            col_0     = image_col_idx - 1
            row_0     = excel_row - 1

            img.width, img.height = IMG_W, IMG_H
            img_w_emu = int(IMG_W * EMU_PER_PIXEL)
            img_h_emu = int(IMG_H * EMU_PER_PIXEL)

            anchor = TwoCellAnchor(
                _from=AnchorMarker(col=col_0, row=row_0, colOff=offset_x, rowOff=offset_y),
                to=AnchorMarker(col=col_0, row=row_0,
                                colOff=offset_x + img_w_emu,
                                rowOff=offset_y + img_h_emu)
            )
            img.anchor = anchor
            sheet.add_image(img, f"{image_col_letter}{excel_row}")
            # ✅ 200px × 0.75 = 150 points (Excel row height is in points)
            sheet.row_dimensions[excel_row].height = 150

    workbook.save(file_path)

## Step 3 · Split by manager and branch

`managers.xlsx` has one column per area manager with that manager's branches listed below it.

In [ ]:
def gather_and_save_data(managers_df, data_df, original_excel_path, sheet_name,
                         output_dir, image_col_name='Images'):
    # Returns {row_num: raw_bytes} — fresh XLImage created per file to avoid stream reuse bugs
    row_image_bytes = get_image_order_from_rich_data(original_excel_path, sheet_name)
    print(f"📸 Ready to place images from {len(row_image_bytes)} rows.")

    for manager in managers_df.columns:
        manager_dir = os.path.join(output_dir, manager)
        os.makedirs(manager_dir, exist_ok=True)
        manager_branches = managers_df[manager].dropna().tolist()
        manager_data     = data_df[data_df['Branch'].isin(manager_branches)]

        for branch in manager_branches:
            branch_data = manager_data[manager_data['Branch'] == branch]
            if len(branch_data) == 0 or branch == 'missing':
                continue

            row_image_mapping = {}
            for new_position, (_, row) in enumerate(branch_data.iterrows()):
                original_row = row['OriginalRow']
                if original_row in row_image_bytes:
                    # Fresh XLImage each time — avoids BytesIO stream exhaustion
                    row_image_mapping[new_position] = XLImage(BytesIO(row_image_bytes[original_row]))
                else:
                    print(f"⚠️  No image for original row {original_row}")

            branch_data_save = branch_data.drop(columns=['OriginalRow'])
            branch_file_path = os.path.join(manager_dir, f'{branch}.xlsx')
            create_formatted_excel(branch_file_path, branch_data_save, row_image_mapping, image_col_name)

    print("✅ Data gathered and saved to folders with images.")

## Run

In [ ]:
managers_df = pd.read_excel(MANAGERS_PATH)

data_df = pd.read_excel(WORKBOOK_PATH, sheet_name=SHEET_NAME)
data_df['OriginalRow'] = data_df.index + 2   # Excel row number, used to match photos

data_day = data_df[data_df['Date'] == REPORT_DATE]

if data_day.empty:
    print(f"⚠️  No rows found for {REPORT_DATE}. Nothing to export.")
else:
    print(f"📅 Found {len(data_day)} rows for {REPORT_DATE}.")
    gather_and_save_data(managers_df, data_day, WORKBOOK_PATH, SHEET_NAME, OUTPUT_DIR, image_col_name='Images')

In [ ]:
# What was written
for manager in sorted(os.listdir(OUTPUT_DIR)):
    manager_dir = os.path.join(OUTPUT_DIR, manager)
    if manager.startswith('.') or not os.path.isdir(manager_dir):
        continue
    files = sorted(f for f in os.listdir(manager_dir) if not f.startswith('.'))
    if files:
        print(f"{manager}/")
        for f in files:
            print(f"    {f}")